In [4]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
true_time=netCDF['time']
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);

#Restricts the timesteps of the data from timesteps0 to 140
data=netCDF.isel(time=np.arange(0,140+1))
parcel=parcel.isel(time=np.arange(0,140+1))
res='1km'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [5]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [23]:
# print('Loading In Data')
# u_data=data['u'].interp(xf=data['xh']).data
# v_data=data['v'].interp(yf=data['yh']).data
# print('done')


# Nt=len(data['time'])
# for t in np.arange(Nt):
#     u_data=data['u'].isel(time=t).interp(xf=data['xh']).data
#     v_data=data['v'].isel(time=t).interp(yf=data['yh']).data
#     print('calculating convergence and taking mean')
#     conv=-(Ddx_3D(u_data,1000)+Ddy_3D(v_data,1000))

In [27]:
if res=='1km':
    dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
if res=='250m':
    dir2='/mnt/lustre/koa/scratch/air673/'
def initiate_array():
    # Define array dimensions (adjust based on your netCDF)
    t_size = len(netCDF['time'])  # Number of timesteps
    z_size = len(netCDF['zh'])    # Number of vertical levels
    y_size = len(netCDF['yh'])    # Number of y-axis points
    x_size = len(netCDF['xh'])    # Number of x-axis points
    
    with h5py.File(dir2 + 'Variable_Calculation/' + 'Convergence'+f'_{res}'+'.h5', 'a') as f:
        # Check if the dataset 'theta_e' already exists
        if 'conv' not in f:
            # Create a dataset with the full size for all time steps (initially empty)
            f.create_dataset('conv', 
                             (t_size, z_size, y_size, x_size),  # Full size for all timesteps
                             maxshape=(None, z_size, y_size, x_size),  # Unlimited timesteps (can grow along time dimension)
                             dtype='float64', 
                             chunks=(1, z_size, y_size, x_size))  # Chunks for time axis to allow resizing

            
def add_timestep_at_index(timestep_data, index):
    with h5py.File(dir2 + 'Variable_Calculation/' + 'Convergence'+f'_{res}'+'.h5', 'a') as f:
        # Access the existing dataset 'theta_e'
        dataset = f['conv']
        
        # Assign the new timestep data at the specified index
        dataset[index] = timestep_data

In [ ]:
#RUNNING

In [33]:
#MAKING ARRAY TO STORE THETA_E
initiate_array()

#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
for t in range(len(netCDF['time'])):
    if np.mod(t,1)==0: print(f'Current time {t}')
    u_data=data['u'].isel(time=t).interp(xf=data['xh']).data
    v_data=data['v'].isel(time=t).interp(yf=data['yh']).data
    print('calculating convergence and taking mean')
    conv=-(Ddx_3D(u_data,1000)+Ddy_3D(v_data,1000))
    add_timestep_at_index(conv, t)


Current time 0
calculating convergence and taking mean
Current time 1
calculating convergence and taking mean
Current time 2
calculating convergence and taking mean
Current time 3
calculating convergence and taking mean
Current time 4
calculating convergence and taking mean
Current time 5
calculating convergence and taking mean
Current time 6
calculating convergence and taking mean
Current time 7
calculating convergence and taking mean
Current time 8
calculating convergence and taking mean
Current time 9
calculating convergence and taking mean
Current time 10
calculating convergence and taking mean
Current time 11
calculating convergence and taking mean
Current time 12
calculating convergence and taking mean
Current time 13
calculating convergence and taking mean
Current time 14
calculating convergence and taking mean
Current time 15
calculating convergence and taking mean
Current time 16
calculating convergence and taking mean
Current time 17
calculating convergence and taking mean
Cu

In [43]:
# dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# file_path = dir2 + 'Variable_Calculation/' + 'Convergence' + f'_{res}' + '.h5'
# with h5py.File(file_path, 'r') as f:
#     Conv = f['conv'][:]